# Mesh — Feedforward Depth vs Splat Renders, then Semantic Query

Fuses the same scene into a TSDF mesh from two depth sources and compares them:

- **feedforward** — the model's own depth maps in `pointcloud.zarr` (`mesh.source: feedforward`, the
  pipeline default, and what `02_pointcloud/feedforward_mesh.ipynb` does).
- **splats** — depth re-rendered from the trained Gaussian splat in `ckpt.pt`, with the render alpha as
  confidence (`mesh.source: splats`). Requires `03_splats/train_splats.ipynb` first.

Both go through `fuse_tsdf`, which takes plain arrays: each source composes its own depth, RGB,
camera-to-world poses and intrinsics, because only the caller knows which resolution grid it is on.
The second half of the notebook replaces the old `Splatter.query_mesh`: lifted MaskCLIP features are
transferred onto the mesh vertices, scored against a text query, and the best-matching connected
region is saved as its own mesh.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import zarr
from zarr.codecs import BloscCodec
import open3d as o3d
import pyvista as pv
from PIL import Image
from tqdm.auto import tqdm
import matplotlib

matplotlib.use("Agg") if os.environ.get("PYVISTA_OFF_SCREEN") else None
%matplotlib inline

%run ../notebook_utils.py
set_notebook_backend()

from collab_splats.geometry.transforms import invert_poses
from collab_splats.mesh import clean_repair_mesh, features2vertex, fuse_tsdf, mesh_clustering
from collab_splats.mesh.io import render_tsdf_inputs
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.pointcloud.utils import confidence_mask, lift_features
from collab_splats.preproc import frames as fr
from collab_splats.semantics.features import BaseQueryableExtractor
from collab_splats.semantics.compression import FeatureAutoencoder
from collab_splats.semantics.utils import ae_path
from collab_splats.utils.visualization import VIZ_KWARGS

## §0 — Configuration

TSDF parameters are the `mesh:` defaults from `configs/base.yaml`; `sdf_trunc` is not a constant here
because `fuse_tsdf` derives it as 4 × `voxel_size`. `CONF_PERCENTILE = 20` masks the lowest 20 % of
feedforward confidence before fusion; the splats path takes no percentile — `render_tsdf_inputs`
already zeroes depth wherever the render alpha is 0. The MaskCLIP cache paths match
`05_lifting/semantic_lifting.ipynb` so a lifted feature set from that notebook is reused.

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
VOXEL_SIZE = 0.005  # base.yaml mesh.voxel_size — sdf_trunc is derived (4 × voxel_size)
DEPTH_TRUNC = 1.0  # base.yaml mesh.depth_trunc (scene units — feedforward depth is not metric)
CONF_PERCENTILE = 20.0

SPLATS_CKPT = OUTPUT_DIR / "splats" / "ckpt.pt"  # written by 03_splats/train_splats.ipynb
MESH_FF_DIR = TUTORIAL_CACHE / "mesh_feedforward"
MESH_SPLATS_DIR = TUTORIAL_CACHE / "mesh_splats"

# Semantic query — the tutorial video (C0043) is an outdoor walk; same queries as nb 05
LATENT_DIM = 13
EXTRACTOR = "maskclip"
QUERY_POSITIVE = ["tree"]
QUERY_NEGATIVE = ["ground"]
QUERY_THRESHOLD = 0.6  # vertices scoring above this seed the clusters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# nb 05 cache layout: compressed per-point codes + the AE that decodes them
_lifted = TUTORIAL_CACHE / "lifted"
AE_MASKCLIP = ae_path(_lifted / "semantics", EXTRACTOR)
LIFTED_MASKCLIP = _lifted / f"lifted_{EXTRACTOR}.zarr"
_lifted.mkdir(parents=True, exist_ok=True)

assert RECON.exists(), f"missing {RECON} — run 02_pointcloud/feedforward_methods.ipynb first"
print(f"RECON:       {RECON}")
print(f"SPLATS_CKPT: {SPLATS_CKPT}  (exists: {SPLATS_CKPT.exists()})")
print(f"Device:      {DEVICE}")

## §1 — Feedforward-source mesh

The default path (`mesh.source: feedforward`) fuses the model's own depth: read `ff.depth` and `ff.images`
straight from the zarr, mask depth with `confidence_mask`, and hand `fuse_tsdf` the model-res
intrinsics that grid belongs to. `load_images=True` is required — the RGB comes from the store.

In [ ]:
# Depth + RGB come from the zarr; poses are the zarr's extrinsics (the pipeline uses COLMAP's)
ff = FeedforwardResult.load_zarr(RECON, load_images=True)
print(f"Loaded: {ff.points.shape[0]:,} pts, {ff.extrinsics.shape[0]} frames, depth {tuple(ff.depth.shape)}")

# Confidence gate first — never fuse depth the model itself is unsure about
depths = np.asarray(ff.depth)
depths = np.where(confidence_mask(np.asarray(ff.confidence), CONF_PERCENTILE), depths, 0.0)

# images is (N, 3, H, W) float in [0, 1]; fuse_tsdf takes (N, H, W, 3) uint8
rgbs = np.asarray(ff.images).transpose(0, 2, 3, 1)
rgbs = np.ascontiguousarray((np.clip(rgbs, 0.0, 1.0) * 255).round().astype(np.uint8))

mesh_ff_path = fuse_tsdf(
    depths,
    rgbs,
    invert_poses(ff.extrinsics),
    ff.intrinsics,
    MESH_FF_DIR,
    voxel_size=VOXEL_SIZE,
    depth_trunc=DEPTH_TRUNC,
)
print(f"Feedforward mesh → {mesh_ff_path}")

## §2 — Splats-source mesh

`render_tsdf_inputs` re-renders `ckpt.pt` into the `(depths, rgbs, c2w, K)` arrays `fuse_tsdf` takes:
rendered depth already zeroed wherever the render alpha is 0, frame-resolution uint8 color, and the
`c2w` the checkpoint was trained with, pose-opt deltas included. The splats stage writes no stored render —
the checkpoint is the artifact, and the views stream one at a time rather than materialising the stack.
Renders are already at frame resolution, so there is no upsampling path.
**Run `03_splats/train_splats.ipynb` first** — the splats stage is never auto-run.

In [ ]:
assert SPLATS_CKPT.exists(), f"missing {SPLATS_CKPT} — run 03_splats/train_splats.ipynb first"

# Rendered depth / RGB / poses straight out of the checkpoint. The renderer already applies
# the alpha gate and returns frame-resolution uint8 color with the poses it rendered from,
# pose-opt deltas included — nothing to lift, nothing to re-pose.
depths_sp, rgbs_sp, c2w_sp, K_sp = render_tsdf_inputs(SPLATS_CKPT)
print(f"depths {depths_sp.shape}  rgbs {rgbs_sp.shape} {rgbs_sp.dtype}  c2w {c2w_sp.shape}  K {K_sp.shape}")

mesh_sp_path = fuse_tsdf(
    depths_sp,
    rgbs_sp,
    c2w_sp,
    K_sp,
    MESH_SPLATS_DIR,
    voxel_size=VOXEL_SIZE,
    depth_trunc=DEPTH_TRUNC,
)
print(f"Splats mesh → {mesh_sp_path}")

## §3 — Compare

Vertex / triangle counts and connected components (Open3D `cluster_connected_triangles`) for both meshes,
then a static side-by-side render. Fewer, larger components means a cleaner surface; the splat-rendered
depth is multi-view consistent by construction, the feedforward depth is per-frame.

In [ ]:
def mesh_to_polydata(mesh: o3d.geometry.TriangleMesh) -> pv.PolyData:
    # Open3D mesh -> PyVista PolyData with uint8 vertex colors
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    faces = np.hstack([np.full((len(triangles), 1), 3), triangles]).ravel()
    poly = pv.PolyData(vertices, faces)
    poly["RGB"] = (np.asarray(mesh.vertex_colors) * 255).astype(np.uint8)
    return poly


def mesh_stats(mesh: o3d.geometry.TriangleMesh) -> dict:
    # Connected components over triangle adjacency
    cluster_ids, cluster_sizes, _ = mesh.cluster_connected_triangles()
    n_components = len(cluster_sizes)
    largest = int(max(cluster_sizes)) if n_components else 0
    return {
        "vertices": len(mesh.vertices),
        "triangles": len(mesh.triangles),
        "components": n_components,
        "largest_component_tris": largest,
    }


meshes = {
    "feedforward": o3d.io.read_triangle_mesh(str(mesh_ff_path)),
    "splats": o3d.io.read_triangle_mesh(str(mesh_sp_path)),
}
stats = {name: mesh_stats(mesh) for name, mesh in meshes.items()}

# Stats table
columns = list(next(iter(stats.values())).keys())
print(f"{'source':<12}" + "".join(f"{col:>24}" for col in columns))
for name, row in stats.items():
    print(f"{name:<12}" + "".join(f"{row[col]:>24,}" for col in columns))

# Side-by-side static render, shared camera
pl = pv.Plotter(shape=(1, 2), window_size=(1600, 700))
for col, (name, mesh) in enumerate(meshes.items()):
    pl.subplot(0, col)
    pl.add_mesh(mesh_to_polydata(mesh), scalars="RGB", rgb=True)
    pl.add_text(f"{name}: {stats[name]['vertices']:,} verts", font_size=12)
    pl.camera_position = [VIZ_KWARGS["position"], VIZ_KWARGS["focal_point"], VIZ_KWARGS["view_up"]]
    pl.camera.azimuth = VIZ_KWARGS["azimuth"]
    pl.camera.elevation = VIZ_KWARGS["elevation"]
    pl.camera.zoom(VIZ_KWARGS["zoom"])
pl.link_views()
pl.show()

## §4 — Pipeline equivalents

- `mesh.source: splats` in the config selects the §2 path inside `Reconstructor.mesh()`; `feedforward`
  (default) selects §1. Both read `mesh.voxel_size` and `mesh.depth_trunc`; `mesh.conf_percentile` is
  feedforward-only, and `sdf_trunc` is not a config key — `fuse_tsdf` derives it as 4 × `voxel_size`.
- Against a processed scene: `--stages splats mesh` trains the splat, then meshes from its renders. The splats
  stage is never auto-run, so `mesh.source: splats` with no `ckpt.pt` on disk raises.
- The pipeline uses COLMAP poses (`colmap/sparse/0`) as the pose authority for the feedforward path; this
  notebook uses the zarr's extrinsics, which are identical unless BA or loop closure ran.

## §5 — Semantic query on the mesh

Replacement for the retired `Splatter.query_mesh`. One step per cell: lifted per-point features →
per-vertex features → text-query scores → painted mesh → largest matching cluster saved as `query_mesh.ply`.

**Step 1 — per-point features.** Same flow as `05_lifting/semantic_lifting.ipynb`: MaskCLIP on every
keyframe, a `FeatureAutoencoder` compressing to `LATENT_DIM`, `lift_features` onto `ff.points`. If nb 05
already wrote its cache under `TUTORIAL_CACHE/lifted/`, it is loaded instead. The AE path moved from
`_lifted/semantics/<extractor>/<extractor>_ae.pt` to `_lifted/semantics/<extractor>_ae.pt`, so a cache
warmed before that change is not found and the AE is refit.

In [ ]:
# MaskCLIP from the registry — used for extraction here and for text scoring below
maskclip = BaseQueryableExtractor.get(EXTRACTOR)(device=DEVICE)

if LIFTED_MASKCLIP.exists() and AE_MASKCLIP.exists():
    # Cache hit: compressed codes + the AE that decodes them back to CLIP space
    ae = FeatureAutoencoder.load(AE_MASKCLIP).to(DEVICE)
    lifted_store = zarr.open_group(store=str(LIFTED_MASKCLIP), mode="r")
    compressed = torch.from_numpy(np.asarray(lifted_store["features"][:]))
    print(f"Loaded from cache: {compressed.shape} → {LIFTED_MASKCLIP}")
else:
    # Pixels come from the canonical images/ dir — ff.image_paths point at a per-run staging dir
    _frames = fr.read_frames(IMAGES_DIR, idxs=[fr.frame_idx_from_path(path) for path in ff.image_paths])
    images = [Image.fromarray(frame).convert("RGB") for frame in tqdm(_frames, desc="Loading frames")]

    # Extract per-frame patch maps, fit a plain AE, compress, lift to the pointcloud
    feature_maps = maskclip.forward(images)  # list of (D, H_p, W_p)
    feature_dim = feature_maps[0].shape[0]
    ae = FeatureAutoencoder(feature_dim, LATENT_DIM).to(DEVICE)
    ae.fit(torch.cat([fmap.flatten(1).T for fmap in feature_maps]).to(DEVICE), epochs=100, target_cosine=0.95)
    compressed_maps = [ae.encode(fmap.to(DEVICE)).detach().cpu() for fmap in feature_maps]
    compressed = lift_features(compressed_maps, ff)  # (P, LATENT_DIM)

    # Persist in nb 05's layout so either notebook can reuse it
    lifted_store = zarr.open_group(store=str(LIFTED_MASKCLIP), mode="w")
    lifted_store.create_array(
        "features", data=compressed.numpy(), chunks=compressed.shape, compressors=BloscCodec(cname="lz4")
    )
    ae.save(AE_MASKCLIP)
    print(f"Saved: {compressed.shape} → {LIFTED_MASKCLIP}")

# Decode to full CLIP dimension — text scoring happens in the decoded space
point_feats = ae.per_point_decode(compressed.to(DEVICE)).detach().cpu().numpy()
print(f"point_feats: {point_feats.shape}")

**Step 2 — vertex features.** `features2vertex` maps point features onto the splats mesh by Gaussian-weighted
KNN (`k=5`); points farther than `sdf_trunc` from their nearest vertex are ignored.

In [ ]:
# Per-vertex features on the splats-source mesh
query_mesh_src = meshes["splats"]
vertices = np.asarray(query_mesh_src.vertices)
vertex_feats = features2vertex(vertices, ff.points, point_feats, k=5, sdf_trunc=0.03)
print(f"vertex_feats: {vertex_feats.shape}")

**Step 3 — scores.** `score_queries` is a contrastive softmax over positive vs negative queries, in `[0, 1]`.

In [ ]:
# Contrastive text score per vertex
vertex_feats_dev = torch.from_numpy(vertex_feats).to(DEVICE)
scores = (
    maskclip.score_queries(vertex_feats_dev, positive=QUERY_POSITIVE, negative=QUERY_NEGATIVE, temperature=0.05)
    .detach()
    .cpu()
    .numpy()
)
n_above = int((scores > QUERY_THRESHOLD).sum())
print(f"scores: {scores.shape}  min={scores.min():.3f}  max={scores.max():.3f}  >{QUERY_THRESHOLD}: {n_above:,}")

**Step 4 — paint.** The mesh colored by score (viridis; yellow = matches `QUERY_POSITIVE`).

In [ ]:
# Score-painted mesh, static render
painted = mesh_to_polydata(query_mesh_src)
painted["semantic"] = scores

pl = pv.Plotter(window_size=(1000, 700))
pl.add_mesh(painted, scalars="semantic", cmap="viridis", clim=(0.0, 1.0), scalar_bar_args={"title": " / ".join(QUERY_POSITIVE)})
pl.camera_position = [VIZ_KWARGS["position"], VIZ_KWARGS["focal_point"], VIZ_KWARGS["view_up"]]
pl.camera.azimuth = VIZ_KWARGS["azimuth"]
pl.camera.elevation = VIZ_KWARGS["elevation"]
pl.camera.zoom(VIZ_KWARGS["zoom"])
pl.show()

**Step 5 — cluster and save.** `mesh_clustering` keeps vertices above `QUERY_THRESHOLD`, connects them within
`spatial_radius`, and returns the connected components (≥ 10 vertices). The largest is cut out with
`select_by_index` and written as `query_mesh.ply` beside the splats mesh.

In [ ]:
# Connected components over high-scoring vertices; keep the largest
clusters = mesh_clustering(query_mesh_src, scores, similarity_threshold=QUERY_THRESHOLD, spatial_radius=0.03)
assert clusters, f"no vertex cluster above {QUERY_THRESHOLD} for {QUERY_POSITIVE} — lower QUERY_THRESHOLD"
largest = max(clusters, key=len)
print(f"{len(clusters)} clusters; largest has {len(largest):,} vertices")

# Sub-mesh of the largest cluster, saved beside the splats mesh
query_mesh = query_mesh_src.select_by_index(largest.tolist())
query_mesh_path = MESH_SPLATS_DIR / "query_mesh.ply"
o3d.io.write_triangle_mesh(str(query_mesh_path), query_mesh)
print(f"query_mesh: {len(query_mesh.vertices):,} verts, {len(query_mesh.triangles):,} tris → {query_mesh_path}")

pl = pv.Plotter(window_size=(1000, 700))
pl.add_mesh(mesh_to_polydata(query_mesh_src), scalars="RGB", rgb=True, opacity=0.15)
pl.add_mesh(mesh_to_polydata(query_mesh), scalars="RGB", rgb=True)
pl.camera_position = [VIZ_KWARGS["position"], VIZ_KWARGS["focal_point"], VIZ_KWARGS["view_up"]]
pl.camera.azimuth = VIZ_KWARGS["azimuth"]
pl.camera.elevation = VIZ_KWARGS["elevation"]
pl.camera.zoom(VIZ_KWARGS["zoom"])
pl.show()